In [20]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from PIL import Image

In [21]:
CSV_PATH = "../processing/color_power_labels.csv"
DATASET_ROOT = "../data/spotlight-sphere-data"

IMG_SIZE = (128, 128)
BATCH_SIZE = 16
np.random.seed(42)
tf.random.set_seed(42)

In [22]:
df = pd.read_csv(CSV_PATH, low_memory=False)

df["image_path"] = df["image_relpath"].astype(str).apply(lambda p: os.path.join(DATASET_ROOT, p))
df = df[df["image_path"].map(os.path.exists)].reset_index(drop=True)
if df.empty:
    raise ValueError("No images found. Check CSV_PATH/DATASET_ROOT/image_relpath values.")

print("Rows with existing images:", len(df))

Rows with existing images: 100


In [23]:
def load_and_preprocess_image(path: str) -> np.ndarray:
    with Image.open(path) as img:
        img = img.convert("RGB").resize(IMG_SIZE, Image.BILINEAR)
        return np.asarray(img, dtype=np.float32) / 255.0

X_images = np.stack([load_and_preprocess_image(p) for p in df["image_path"]], axis=0)
print("Image tensor shape:", X_images.shape)

Image tensor shape: (100, 128, 128, 3)


In [ ]:
color_cols = ["light_color_r", "light_color_g", "light_color_b"]
energy_col = "light_energy"
y_color_raw = df[color_cols].to_numpy(dtype=np.float32)
y_energy_log_raw = np.log(df[energy_col].to_numpy(dtype=np.float32)).reshape(-1, 1)
idx = np.arange(len(df))
idx_train, idx_test = train_test_split(idx, test_size=0.2, random_state=42)
idx_train, idx_val = train_test_split(idx_train, test_size=0.2, random_state=42)
X_train, X_val, X_test = X_images[idx_train], X_images[idx_val], X_images[idx_test]
y_color_train_raw, y_color_val_raw, y_color_test_raw = y_color_raw[idx_train], y_color_raw[idx_val], y_color_raw[idx_test]
y_energy_train_raw, y_energy_val_raw, y_energy_test_raw = y_energy_log_raw[idx_train], y_energy_log_raw[idx_val], y_energy_log_raw[idx_test]

color_mean = y_color_train_raw.mean(axis=0, keepdims=True)
color_std = y_color_train_raw.std(axis=0, keepdims=True)
color_std[color_std < 1e-8] = 1.0
energy_mean = y_energy_train_raw.mean(axis=0, keepdims=True)
energy_std = y_energy_train_raw.std(axis=0, keepdims=True)
energy_std[energy_std < 1e-8] = 1.0
y_color_train = (y_color_train_raw - color_mean) / color_std
y_color_val = (y_color_val_raw - color_mean) / color_std
y_color_test = (y_color_test_raw - color_mean) / color_std
y_energy_train = (y_energy_train_raw - energy_mean) / energy_std
y_energy_val = (y_energy_val_raw - energy_mean) / energy_std
y_energy_test = (y_energy_test_raw - energy_mean) / energy_std
print("Train/Val/Test:", len(idx_train), len(idx_val), len(idx_test))


Train/Val/Test: 64 16 20


In [25]:
img_input = keras.Input(shape=(*IMG_SIZE, 3), name="image")
x = layers.Conv2D(32, 3, activation="relu", padding="same")(img_input)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)

color_out = layers.Dense(3, name="color_head")(x)
energy_out = layers.Dense(1, name="energy_head")(x)

model = keras.Model(inputs=img_input, outputs=[color_out, energy_out])
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss={"color_head": "mse", "energy_head": "mse"},
    loss_weights={"color_head": 1.0, "energy_head": 0.7},
    metrics={"color_head": ["mae"], "energy_head": ["mae"]},
)
model.summary()

callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True)]

history = model.fit(
    X_train,
    {"color_head": y_color_train, "energy_head": y_energy_train},
    validation_data=(X_val, {"color_head": y_color_val, "energy_head": y_energy_val}),
    epochs=60,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)  │ (None, 128, 128,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 128, 128,  │        896 │ image[0][0]       │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 64, 64,    │          0 │ conv2d_6[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 64, 64,    │     18,496 │ max_pooling2d_4[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 32, 32,    │          0 │ conv2d_7[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 32, 32,    │     73,856 │ max_pooling2d_5[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ conv2d_8[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     16,512 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ color_head (Dense)  │ (None, 3)         │        387 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ energy_head (Dense) │ (None, 1)         │        129 │ dropout_2[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 110,276 (430.77 KB)

 Trainable params: 110,276 (430.77 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/60
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 157ms/step - color_head_loss: 1.0024 - color_head_mae: 0.8595 - energy_head_loss: 0.9924 - energy_head_mae: 0.8752 - loss: 1.6971 - val_color_head_loss: 1.2456 - val_color_head_mae: 0.9444 - val_energy_head_loss: 0.8644 - val_energy_head_mae: 0.8284 - val_loss: 1.8507
Epoch 2/60
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step - color_head_loss: 0.9926 - color_head_mae: 0.8552 - energy_head_loss: 0.9122 - energy_head_mae: 0.8315 - loss: 1.6311 - val_color_head_loss: 1.2507 - val_color_head_mae: 0.9464 - val_energy_head_loss: 0.7615 - val_energy_head_mae: 0.7787 - val_loss: 1.7838
Epoch 3/60
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step - color_head_loss: 0.9849 - color_head_mae: 0.8565 - energy_head_loss: 0.8521 - energy_head_mae: 0.7796 - loss: 1.5813 - val_color_head_loss: 1.2536 - val_color_head_mae: 0.9424 - val_energy_head_loss: 0.7071 - val_energy_head_mae: 0.7374 - val_loss: 1.7486
Epoch 4/60
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - color_head_loss: 0.9812 

In [ ]:
metrics = model.evaluate(
    X_test,
    {"color_head": y_color_test, "energy_head": y_energy_test},
    verbose=0,
)
print("Raw Keras test metrics:", metrics)

pred_color_z, pred_energy_z = model.predict(X_test, verbose=0)

# invert standardization
pred_color = pred_color_z * color_std + color_mean
true_color = y_color_test_raw

pred_energy_log = pred_energy_z * energy_std + energy_mean
true_energy_log = y_energy_test_raw

pred_color = np.clip(pred_color, 0.0, 1.0)
color_mae = np.mean(np.abs(pred_color - true_color), axis=0)
energy_log_mae = np.mean(np.abs(pred_energy_log[:, 0] - true_energy_log[:, 0]))

# convert back to original units
pred_energy = np.exp(pred_energy_log[:, 0])
true_energy = np.exp(true_energy_log[:, 0])
energy_mae = np.mean(np.abs(pred_energy - true_energy))
energy_mape = np.mean(np.abs(pred_energy - true_energy) / np.clip(true_energy, 1e-6, None)) * 100.0

print("Color MAE [r, g, b]:", color_mae)
print(f"Energy MAE (log space): {energy_log_mae:.6f}")
print(f"Energy MAE (original units): {energy_mae:.3f}")
print(f"Energy MAPE (%): {energy_mape:.2f}")


Raw Keras test metrics: [0.6883381009101868, 0.614671528339386, 0.1052379384636879, 0.5945636630058289, 0.2312450408935547]
Color MAE [r, g, b]: [0.19720891 0.14976801 0.11339702]
Energy MAE (log space): 0.325299
Energy MAE (original units): 5001.232
Energy MAPE (%): 27.55


In [ ]:
# save model for later testing
model.save("color_power_predictor.keras")

# single image inference + comparison
sample_idx = int(idx_test[0])
sample_path = df.iloc[sample_idx]["image_path"]
sample_relpath = df.iloc[sample_idx]["image_relpath"]

sample_img = np.expand_dims(load_and_preprocess_image(sample_path), axis=0)
pred_color_z, pred_energy_z = model.predict(sample_img, verbose=0)

pred_color = np.clip((pred_color_z * color_std + color_mean)[0], 0.0, 1.0)
pred_energy_log = (pred_energy_z * energy_std + energy_mean)[0, 0]
pred_energy = float(np.exp(pred_energy_log))

true_row = df.iloc[sample_idx]
true_color = true_row[color_cols].to_numpy(dtype=np.float32)
true_energy = float(true_row[energy_col])
true_energy_log = float(np.log(true_energy))

print("Image:", sample_relpath)
print("Pred color [r,g,b]:", pred_color)
print("True color [r,g,b]:", true_color)
print(f"Pred energy (log): {pred_energy_log:.6f}")
print(f"True energy (log): {true_energy_log:.6f}")
print("Pred energy:", pred_energy)
print("True energy:", true_energy)


Image: 0083.png
Pred color [r,g,b]: [0.73837715 0.44946313 0.70144206]
True color [r,g,b]: [0.8547057  0.55361485 0.7670229 ]
Pred energy (log): 9.121878
True energy (log): 8.914887
Pred energy: 9153.3720703125
True energy: 7441.941156205622
